In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import sys
sys.path.append("..")  # helpers.py и prompt_evaluator.py лежат в корне курса
from helpers import add_user_message, chat
from prompt_evaluator import PromptEvaluator

client = Anthropic()
model = "claude-haiku-5-5"
# Без thinking: ответы плана питания не упираются в max_tokens
thinking = {"type": "disabled"}

In [ ]:
# Больше потоков — быстрее, но можно получить ошибку rate limit
evaluator = PromptEvaluator(client, model, max_concurrent_tasks=3)

In [ ]:
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan "
    "for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    output_file="15_dataset.json",
    num_cases=3,
)

In [ ]:
def run_prompt(prompt_inputs, system=None, requirements=""):
    """Простой prompt; requirements — что должно быть в ответе"""
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
{requirements}"""
    messages = []
    add_user_message(messages, prompt)
    return chat(
        client, model, messages, system=system, thinking=thinking
    )

In [ ]:
# По этим требованиям grader ставит балл
extra_criteria = """
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""

# system prompt для коротких ответов
system_prompt = """
Write short, plain answers. The reader wants facts, not style.
- Use simple, common words.
- Give only what the request asks for. Skip introductions,
  summaries and advice that was not requested.
- Write in short sentences, without adjectives and adverbs
  that add no information.
- Prefer lists and tables to paragraphs.
"""

In [ ]:
# 1. Простой prompt — базовый балл
results_base = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="15_dataset.json",
    extra_criteria=extra_criteria,
    json_output_file="15_output_base.json",
    html_output_file="15_output_base.html",
)

In [ ]:
# 2. + system prompt для коротких ответов
results_short = evaluator.run_evaluation(
    run_prompt_function=lambda inputs: run_prompt(inputs, system_prompt),
    dataset_file="15_dataset.json",
    extra_criteria=extra_criteria,
    json_output_file="15_output_short.json",
    html_output_file="15_output_short.html",
)

In [ ]:
# 3. + требования к ответу прямо в prompt
results_criteria = evaluator.run_evaluation(
    run_prompt_function=lambda inputs: run_prompt(
        inputs, system_prompt, extra_criteria
    ),
    dataset_file="15_dataset.json",
    extra_criteria=extra_criteria,
    json_output_file="15_output_criteria.json",
    html_output_file="15_output_criteria.html",
)